In [7]:
import json
import os

import pandas as pd

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")
     

Client ready.


In [8]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is doppler effect?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

The Doppler effect is a phenomenon in physics that describes how the frequency of a wave changes when its source is moving relative to an observer. This effect is named after Christian Doppler, who first described it in 1842.

The Doppler effect occurs when a wave, such as sound or light, is emitted by a source that is moving towards or away from an observer. When the source is moving towards the observer, the frequency of the wave increases, and when it is moving away, the frequency decreases.

Here's a simple explanation:

1. **Source moving towards the observer**: The waves are compressed, resulting in a higher frequency (or pitch) and a shorter wavelength.
2. **Source moving away from the observer**: The waves are stretched, resulting in a lower frequency (or pitch) and a longer wavelength.

The Doppler effect is commonly observed in various situations, such as:

* **Police sirens**: When a police car is approaching, its siren sounds higher pitched, and when it's moving away, the p

## Student Reasoning — Anatomy of a Call

**1. Difference between system and user**

- **system**: Tells the AI how to behave or what role to take.
  - Example: `"You are a helpful math tutor."`
- **user**: Contains the actual question or instruction we want the AI to respond to.
  - Example: `"Simple explain what eigenvalues and eigenvectors are."`

**2. What is a token?**

A **token** is a small piece of text that an LLM processes, such as a word, or part of a word.

**Why do API providers bill per token rather than per request?**

API providers bill per token because **different requests use different amounts of text**. Tokens provide a better measure of how much the model processes and generates.

In [9]:
# TODO: Ask the SAME question 5 times at temperature=0.0
# and 5 times at temperature=1.2.

# TODO: Print all 10 answers, grouped by temperature.
question = "What is core banking system?"

print("--- Temperature 0.0 ---")
for i in range(5):
  answer = ask_llm(question, temperature=0.0)
  print(f"{i + 1}. {answer.choices[0].message.content}")

print("\n--- Temperature 1.2 ---")
for i in range(5):
  answer = ask_llm(question, temperature=1.2)
  print(f"{i + 1}. {answer.choices[0].message.content}")

--- Temperature 0.0 ---
1. A core banking system (CBS) is a software application that enables banks and other financial institutions to manage their daily operations, customer accounts, and financial transactions efficiently. It is the backbone of a bank's information technology (IT) infrastructure, providing a centralized platform for processing transactions, managing accounts, and delivering banking services to customers.

A core banking system typically includes the following features:

1. **Account management**: CBS allows banks to create, manage, and maintain customer accounts, including deposit accounts, loan accounts, and credit card accounts.
2. **Transaction processing**: CBS enables banks to process various types of transactions, such as deposits, withdrawals, transfers, and payments.
3. **Payment processing**: CBS facilitates payment processing, including check clearing, electronic funds transfer (EFT), and online payment processing.
4. **Loan management**: CBS provides tool

## Student Reasoning — Temperature

- **Temperature = 0.0:** The answers were very similar. The model consistently explained core banking systems using almost the same structure and ideas.


- **Temperature = 1.2:** The answers were more varied. The model used different wording, examples, and structures, and one response even reached the `max_tokens` limit.

For the **loan decision-support system**, a **low temperature (around 0.0–0.3)** is more appropriate because decisions should be consistent, predictable, and less influenced by randomness.

In [10]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [11]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
SUMMARY_PROMPT_V1 = "Summarize this:"

#   Run it on L002 and L006. Read the output critically.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications factually and neutrally.
Do not invent or assume any details.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    v1 = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    
    v2 = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )

    print(f"\n========== {letter_id} ==========")
    print("\nV1:")
    print(v1.choices[0].message.content)
    
    print("\nV2:")
    print(v2.choices[0].message.content)


--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season, and is willing to repay the loan when he can. However, he currently has no collateral to offer.

--- L006 ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, with no collateral offered, relying on his personal guarantee of being "trustworthy".

--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business

## Student Reasoning — Summarization Prompts

**1. Problems with V1 and how V2 fixed them**

- **V1 sometimes added assumptions.** For L006, V1 said Kofi had **"no prior experience"**, but the letter only says he has not started the businesses yet.
- V1 also added **"as assurance"** to Kofi's trustworthiness, which was not explicitly stated.
- **V2 was more factual and neutral**, avoided unnecessary assumptions, and followed the 3 to 4 sentence constraint.

**2. Why is "no invented details" important?**

Loan decisions should be based only on the applicant's actual information. Invented details could unfairly affect a person's loan decision.

This failure mode is called **hallucination** in LLMs.

In [12]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_SYSTEM_PROMPT = """You extract structured information from loan applications.

Return ONLY a valid JSON object with EXACTLY these keys:
{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

Rules:
- If a field is not stated in the letter, use null.
- Do not guess or invent information.
- amount_ghs, monthly_profit_ghs, and repayment_months must be numbers.
- has_collateral_or_guarantor must be true or false.
- Return ONLY the JSON object. No explanation or markdown.

Example:

Letter:
"My name is Ama Asante. I need GHS 5,000 to buy stock for my shop.
I make GHS 700 profit per month and can repay in 10 months.
My brother will guarantee the loan."

JSON:
{
  "applicant_name": "Ama Asante",
  "amount_ghs": 5000,
  "purpose": "buy stock for my shop",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}
"""

EXTRACT_PROMPT = """Extract the required information from this loan application.

Loan application:
{letter_text}
"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0.0
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    result = result.removesuffix("```")

    result = result.strip()

    try:
        data = json.loads(result)
        return data
    except json.JSONDecodeError:
        print("Warning: Could not parse LLM response as JSON.")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df = pd.DataFrame(results)

# Put letter_id first
columns = ["letter_id"] + [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

df = df[columns]

display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


## Student Reasoning — Structured Extraction

**1. Why must the few-shot example be different?**

Using one of the six letters could make the model memorize or copy information from the data being evaluated. A separate example tests whether the prompt works generally.

**2. Why use `null, do not guess`?**

It prevents the model from making up missing information. Without this instruction, the model may try to **infer or invent** values that were not stated.

**3. Why use temperature = 0?**

For extraction, we want **consistent and predictable results**, so temperature `0` is appropriate. For creative tasks, a higher temperature can be better because it produces more varied and creative responses.

In [15]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer.

Your task is to analyze a loan application and provide a factual decision-support brief.

Your response must contain exactly these four sections:

1. Strengths
- Bullet points grounded only in the letter.

2. Risks / red flags
- Bullet points based only on information in the letter.

3. Missing information
- List important information the loan officer should request.

4. Suggested next step
- Suggest an action such as "invite for interview", "request documents",
  or "flag for senior review".
- Do NOT recommend approving or rejecting the loan.

Important:
- Do not invent or assume information.
- Clearly distinguish stated facts from missing information.
- Final loan decisions are made by human loan officers, not by the AI.
"""

BRIEF_PROMPT = """Analyze this loan application using the extracted information below.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}

for letter_id, letter_text in LETTERS.items():
    row = df[df["letter_id"] == letter_id]

    if row.empty:
        briefs[letter_id] = "Extraction failed; no brief generated."
        continue

    extracted_json = row.iloc[0].to_dict()

    response = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=json.dumps(extracted_json, indent=2)
        ),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0
    )

    briefs[letter_id] = response.choices[0].message.content
    
# Print the briefs for L001, L002, and L006
for letter_id in ["L001", "L002", "L006"]:
    print()
    print(f"==== BRIEF — {letter_id} ====")
    print()
    print(briefs[letter_id])


==== BRIEF — L001 ====

## Step 1: Strengths
- The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating stability and knowledge in her business.
- She has a steady monthly profit of GHS 900 from her current stall.
- Akosua has saved GHS 2,500 over two years with the susu scheme, demonstrating her ability to save and commit to financial obligations.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.
- The applicant has a clear plan for repayment, proposing to pay GHS 450 monthly over 20 months.

## Step 2: Risks / red flags
- The loan amount of GHS 8,000 is significant compared to her monthly profit and savings, which might pose a risk if her business expansion does not generate enough additional income.
- There is no detailed information on how the expansion into frozen foods will affect her current business operations or profitability.
- The repayment plan relies on her 

## Student Reasoning — Decision Support

**1. Compare L003 and L006**

The system correctly identified the main differences:

- **L003:** It identified strong business experience, regular profit, savings history, and a fixed deposit as strengths. It also suggested verifying the business plan and supporting documents.
- **L006:** It identified major risks such as no existing business, no collateral, no proven cash flow, and an uncertain repayment plan.

Overall, the briefs correctly reflected that **L003 is stronger while L006 carries much higher risk**.

**2. Why forbid "approve/reject"?**

- **Practical reason:** The AI can make mistakes or miss important information, so a human loan officer should make the final decision.
- **Ethical reason:** Loan decisions can seriously affect people's lives, so giving an AI full authority could lead to unfair or biased decisions.

### Part 3.4 — Commit your prompt templates
Commit hash: 4f711bf48aecf088a423c412d3b0986ef95d028c

In [16]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).


# Compare extracted values with GOLD values field by field.
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    row = {"field": field}
    correct = 0

    for letter_id in GOLD:
        predicted = df.loc[df["letter_id"] == letter_id, field].iloc[0]
        expected = GOLD[letter_id][field]

        # Name comparison is case-insensitive
        if field == "applicant_name":
            match = str(predicted).strip().lower() == str(expected).strip().lower()
        else:
            match = predicted == expected

        row[letter_id] = "✓" if match else "✗"

        if match:
            correct += 1

    row["accuracy"] = f"{correct}/3 ({correct / 3:.0%})"
    rows.append(row)

accuracy_df = pd.DataFrame(rows)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3 (100%)
1,amount_ghs,✓,✓,✓,3/3 (100%)
2,purpose,✗,✗,✗,0/3 (0%)
3,monthly_profit_ghs,✓,✓,✗,2/3 (67%)
4,has_collateral_or_guarantor,✓,✓,✓,3/3 (100%)
5,repayment_months,✓,✓,✓,3/3 (100%)


In [17]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def extract_fields(letter_text, temperature=0.0):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=temperature
    )

    result = response.choices[0].message.content.strip()

    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    result = result.removesuffix("```")

    try:
        return json.loads(result.strip())
    except json.JSONDecodeError:
        print("Warning: Could not parse LLM response as JSON.")
        return None

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
for temperature in [0.0, 1.0]:
    results = []

    for _ in range(5):
        result = extract_fields(LETTERS["L004"], temperature=temperature)
        results.append(result)

    valid_results = [r for r in results if r is not None]

    unique_results = {
        json.dumps(r, sort_keys=True)
        for r in valid_results
    }

    print(f"\nTemperature = {temperature}")
    print(f"Valid JSON: {len(valid_results)}/5")
    print(f"Identical values across runs: {len(unique_results) == 1}")
    print(f"Unique result sets: {len(unique_results)}")


Temperature = 0.0
Valid JSON: 5/5
Identical values across runs: True
Unique result sets: 1

Temperature = 1.0
Valid JSON: 5/5
Identical values across runs: True
Unique result sets: 1


In [18]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
# Test 1: Ask for information that is NOT in the letter.

adversarial_summary_prompt = """Summarize this loan application.
Also state the applicant's credit score.

Loan application:
{letter_text}
"""

response = ask_llm(
    adversarial_summary_prompt.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0
)

test1_output = response.choices[0].message.content

# Test 2: Give the extractor completely irrelevant text.

weather_report = """
Today's weather will be partly cloudy with temperatures around 29°C.
There is a chance of rain in the afternoon, with moderate winds expected.
"""

test2_result = extract_fields(weather_report)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("TEST 1: Missing Information")
print(test1_output)

print()

print("\nTEST 2: Irrelevant Input")
print(test2_result)


TEST 1: Missing Information
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He mentions that business has been slow but expects it to improve after the festive season. The applicant does not have collateral to offer at the moment. The credit score of the applicant is not provided in the loan application.


TEST 2: Irrelevant Input
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


## Student Reasoning — Evaluation Results

**1. Extraction accuracy**

The overall accuracy was **78% (14/18 fields)**.

- **100%:** applicant name, loan amount, collateral/guarantor, repayment period.
- **67%:** monthly profit.
- **0%:** purpose, which was the hardest field. The model likely struggled because the purpose descriptions in the letters did not exactly match the wording expected in the `GOLD` labels.

**2. Reliability and temperature**

Both temperatures produced **5/5 valid JSON results with identical values across all runs**. This shows that the extraction was stable in this test. However, production systems should still prefer **low temperature** for predictable and consistent structured extraction.

**3. Hallucination testing**

The system **passed both adversarial tests**. It admitted that the credit score was not provided and returned `null` for all fields when given an irrelevant weather report.

To reduce hallucination risk further, we can use explicit **"do not guess, use null"** instructions, strict JSON schemas, validation rules, and human review for important decisions.

## Student Reasoning — Appropriateness

**1. Who could be unfairly harmed?**

Applicants who write poorly in English could be unfairly judged even if they have strong businesses. The AI may misunderstand their applications and identify incorrect risks, leading to unfair loan decisions.

**2. What are the implications of using a third-party API?**

Sending loan letters to a third-party API could expose sensitive personal and financial information. Before deployment, I would check Ghanaian data-protection requirements, the provider's privacy and data-retention policies, where the data is stored, and whether the institution has permission to transfer the data outside Ghana.

**3. Two production safeguards**

1. **Human review:** An AI recommendation should never automatically approve or reject a loan. A qualified loan officer must review the evidence and make the final decision.
2. **Monitoring and appeals:** Log AI outputs and decisions for auditing, monitor for errors or bias, and provide applicants with a way to challenge or appeal decisions.

## Section 5 — Reflection

### 1. Prompting as engineering

Prompt iteration is similar to tuning model hyperparameters because both involve testing, evaluating results, and adjusting settings to improve performance. The difference is that prompt engineering changes the **instructions and context given to the model**, while hyperparameter tuning changes how the model itself learns or behaves during training.

### 2. Trust

I would **not trust the system to run unattended** because the extraction evaluation achieved only **78% accuracy**, with the `purpose` field scoring **0%**. This shows that even when the system produces valid and consistent output, it can still extract important information incorrectly.

### 3. Cost and scale

From my API usage, one call used roughly **500 tokens**. At that rate, 1,000 applications would require about **500,000 tokens** for one LLM call per application. If multiple calls are needed for each application, usage would increase significantly, so provider pricing, token limits, reliability, and data privacy would be important factors when choosing a provider.

### 4. Looking back at the course

For this task, calling a foundation-model API is better than training our own model because it provides strong language understanding without requiring a large dataset, expensive training, and significant computing resources. However, training our own model may be better when we need **full control, specialized behavior, lower long-term costs at very large scale, or strict data privacy**.